# Reproduce every result from scratch

This notebook regenerates all experiment outputs of *Calibrate Where You Deploy: Conformal Sensor Escalation Under Operating-Point Shift* from the released feature files, then compares them with the CSVs, tables and macros committed in the repository. It is the end-to-end check; `01_kaist_conformal_gate.ipynb` and `02_paderborn_replication.ipynb` only read the committed CSVs.

Runtime on a free Google Colab CPU instance: about 45–60 minutes (KAIST ~3 min, Paderborn E1–E6 ~25 min, Paderborn X1–X3 ~20 min). Keep the tab open.

Pipeline: `data/*.parquet` (features) → `run_experiments.py` → `results/`, `results_paderborn/` → `run_paderborn_extra.py` → extra CSVs → `paper/make_tables_ifac.py` → `paper/tables_ifac/*.tex`, `paper/numbers_ifac.tex`. Feature extraction from the raw signals is in `extract_kaist.py`, `extract_paderborn.py` and `paderborn_colab.ipynb` and is not repeated here (the raw archives are large; the released parquets are the inputs of every experiment).

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/candidoalfilho/conformal_escalation"
if not os.path.exists("cmlib.py"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "conformal_escalation"], check=True)
    os.chdir("conformal_escalation")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
print("working directory:", os.getcwd())

## 1. KAIST — experiments E1–E6 (~3 min)

In [ ]:
!python run_experiments.py --dataset kaist

## 2. Paderborn — experiments E1–E6 (~25 min)

In [ ]:
!python run_experiments.py --dataset paderborn

## 3. Paderborn — extra experiments X1–X3 (~20 min)

In [ ]:
!python run_paderborn_extra.py

## 4. Regenerate the manuscript tables and macros

In [ ]:
!python paper/make_tables_ifac.py

## 5. Compare with the committed results

An empty diff means every CSV, table and headline macro reproduced exactly. Differences in the third decimal of random-forest quantities can arise from BLAS/threading differences between machines and do not change any conclusion; anything larger should be reported as an issue.

In [ ]:
print("=== files that differ from the committed versions (empty = exact reproduction) ===")
!git status --short -- results results_paderborn paper/tables_ifac paper/numbers_ifac.tex
print()
print("=== headline macros ===")
!git diff --stat -- paper/numbers_ifac.tex paper/tables_ifac
!git diff -- paper/numbers_ifac.tex